In [ ]:
from azure.storage.blob import BlobServiceClient
import pandas as pd
import numpy as np
import os
from io import BytesIO

connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
container_name = "processeddata"

blob_service_client = BlobServiceClient.from_connection_string(connection_string)

In [2]:
def load_blob_csv(blob_name):
    blob_client = blob_service_client.get_blob_client(
        container=container_name,
        blob=blob_name
    )

    data = blob_client.download_blob().readall()
    return pd.read_csv(BytesIO(data))

# EKSEMPEL (tilpass til dine faktiske filer)
df_consumption = load_blob_csv("BREIVE_processed.csv")
df_weather = load_blob_csv("BREIVE_weather.csv")
df_norgespris = load_blob_csv("breive_norgespris.csv")

In [3]:
print(df_consumption.head())
print(df_weather.head())
print(df_norgespris.head())

               metering_point_anonymous            timestamp  value_kwh  \
0  fcf4855f-feca-467b-bad2-ff524b7fdca3  2025-01-17 23:00:00       5.63   
1  fcf4855f-feca-467b-bad2-ff524b7fdca3  2025-01-18 00:00:00       4.21   
2  fcf4855f-feca-467b-bad2-ff524b7fdca3  2025-01-18 01:00:00       3.34   
3  fcf4855f-feca-467b-bad2-ff524b7fdca3  2025-01-18 02:00:00       3.82   
4  fcf4855f-feca-467b-bad2-ff524b7fdca3  2025-01-18 03:00:00       3.04   

  transformer_station  consumption_code  hour  weekday  month  is_weekend  \
0              BREIVE                36    23        4      1       False   
1              BREIVE                36     0        5      1        True   
2              BREIVE                36     1        5      1        True   
3              BREIVE                36     2        5      1        True   
4              BREIVE                36     3        5      1        True   

   is_holiday day_night  norgespris  
0       False     night           0  
1       Fa

In [18]:
import numpy as np
import pandas as pd
from linearmodels.panel import PanelOLS


def prepare_norgespris_regression_data(df_consumption, df_weather=None, df_norgespris=None):
    df = df_consumption.copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"])

    if "norgespris" not in df.columns:
        raise ValueError("df_consumption må inneholde kolonnen 'norgespris'.")

    df = df[df["norgespris"].notna()].copy()
    df["norgespris"] = df["norgespris"].astype(int)

    if df_weather is not None:
        weather_df = df_weather.copy()
        weather_df["timestamp"] = pd.to_datetime(weather_df["timestamp"])
        df = df.merge(weather_df, on="timestamp", how="left")

    if df_norgespris is not None and {"transformer_station", "count_total"}.issubset(df_norgespris.columns):
        station_df = df_norgespris.copy()
        station_df["timestamp"] = pd.to_datetime(station_df["timestamp"])
        station_df["date"] = station_df["timestamp"].dt.normalize()

        df["date"] = df["timestamp"].dt.normalize()
        station_df = station_df[["date", "transformer_station", "count_total"]].drop_duplicates()
        df = df.merge(station_df, on=["date", "transformer_station"], how="left")

    return df


def fit_best_norgespris_model(df, dependent_variable="value_kwh"):
    required_columns = {
        "metering_point_anonymous",
        "timestamp",
        dependent_variable,
        "norgespris",
        "air_temperature",
        "wind_speed",
        "precipitation_mm",
        "hour",
        "is_weekend",
        "month",
        "is_holiday",
    }

    missing_columns = required_columns.difference(df.columns)
    if missing_columns:
        missing_text = ", ".join(sorted(missing_columns))
        raise ValueError(f"Datasettet mangler nødvendige kolonner: {missing_text}")

    modeling_df = df.copy()
    modeling_df["timestamp"] = pd.to_datetime(modeling_df["timestamp"])
    modeling_df["metering_point_anonymous"] = modeling_df["metering_point_anonymous"].astype(str)
    modeling_df["is_weekend"] = modeling_df["is_weekend"].astype(int)
    modeling_df["is_holiday"] = modeling_df["is_holiday"].astype(int)
    modeling_df["hour"] = modeling_df["hour"].astype(int)
    modeling_df["month"] = modeling_df["month"].astype(int)
    modeling_df["log_value_kwh"] = np.log1p(modeling_df[dependent_variable])

    modeling_df = modeling_df.dropna(
        subset=[
            dependent_variable,
            "norgespris",
            "air_temperature",
            "wind_speed",
            "precipitation_mm",
            "hour",
            "is_weekend",
            "month",
            "is_holiday",
            "metering_point_anonymous",
        ]
    ).copy()

    panel_df = modeling_df.set_index(["metering_point_anonymous", "timestamp"]).sort_index()

    formula = (
        "log_value_kwh ~ 1 + norgespris + air_temperature + wind_speed + precipitation_mm "
        "+ is_weekend + is_holiday + C(hour) + C(month) + EntityEffects"
    )

    model = PanelOLS.from_formula(formula, data=panel_df, drop_absorbed=True)
    results = model.fit(cov_type="clustered", cluster_entity=True, cluster_time=True)

    metrics = {
        "nobs": int(results.nobs),
        "r2_within": float(results.rsquared_within),
        "policy_effect_pct": float((np.exp(results.params.get("norgespris", 0.0)) - 1) * 100),
    }

    return results, modeling_df, metrics


regression_df = prepare_norgespris_regression_data(df_consumption, df_weather, df_norgespris)
results, analysis_df, metrics = fit_best_norgespris_model(regression_df)

print(f"Observasjoner totalt: {analysis_df.shape[0]:,}")
print(f"Brukt i modell: {metrics['nobs']:,}")
print(f"R2 (within): {metrics['r2_within']:.4f}")
print(f"Estimert policy-effekt: {metrics['policy_effect_pct']:.2f}%")
print()
print(results.summary)
print()
print(f"Norgespris-koeffisient (log-skala): {results.params.get('norgespris'):.4f}")
print(f"Standardfeil: {results.std_errors.get('norgespris'):.4f}")
print(f"p-verdi: {results.pvalues.get('norgespris'):.4g}")

Observasjoner totalt: 5,479,690
Brukt i modell: 5,479,690
R2 (within): 0.1634
Estimert policy-effekt: 2.23%

                          PanelOLS Estimation Summary                           
Dep. Variable:          log_value_kwh   R-squared:                        0.1634
Estimator:                   PanelOLS   R-squared (Between):              0.0000
No. Observations:             5479690   R-squared (Within):               0.1634
Date:                Tue, Apr 14 2026   R-squared (Overall):              0.0994
Time:                        14:03:55   Log-likelihood                -2.182e+06
Cov. Estimator:             Clustered                                           
                                        F-statistic:                   3.451e+04
Entities:                        1322   P-value                           0.0000
Avg Obs:                       4145.0   Distribution:              F(31,5478337)
Min Obs:                       4145.0                                           


In [19]:
print('Nøkkeltall fra regresjonen')
print(f"Observasjoner i modell: {metrics['nobs']:,}")
print(f"R2 (within): {metrics['r2_within']:.4f}")
print(f"Norgespris-koeffisient: {results.params.get('norgespris'):.6f}")
print(f"Omtrent policy-effekt: {(np.exp(results.params.get('norgespris')) - 1) * 100:.2f}%")
print(f"Standardfeil: {results.std_errors.get('norgespris'):.6f}")
print(f"p-verdi: {results.pvalues.get('norgespris'):.6g}")

Nøkkeltall fra regresjonen
Observasjoner i modell: 5,479,690
R2 (within): 0.1634
Norgespris-koeffisient: 0.022012
Omtrent policy-effekt: 2.23%
Standardfeil: 0.004816
p-verdi: 4.87059e-06


In [20]:
mean_log_value_kwh = np.log1p(analysis_df['value_kwh']).mean()
coef = results.params.get('norgespris')
percent_change = (np.exp(coef) - 1) * 100

print(f'Gjennomsnittlig log1p(value_kwh): {mean_log_value_kwh:.4f}')
print(f'Beregnet policy-effekt: {percent_change:.2f}%')

Gjennomsnittlig log1p(value_kwh): 0.8489
Beregnet policy-effekt: 2.23%
